# Katelyn v1 Mirror Difference

Create a `PrototypeOptimal` model with the same architecture hyperparameters as `configs/runs/optimal-prototype/katelyn-v1-images.yaml`, load the bundled test image from `hippy2d.utils.get_testing_img`, and compare mirrored versions of the image.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import ImageOps

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and (cwd.parent / "hippy2d").exists():
    project_dir = cwd.parent
elif (cwd / "hippy2d" / "hippy2d").exists():
    project_dir = cwd / "hippy2d"
elif (cwd / "hippy2d").exists():
    project_dir = cwd
else:
    raise RuntimeError(f"Could not locate the hippy2d project from {cwd}")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print(project_dir)

## Model

These are the `m_param` values from `katelyn-v1-images.yaml`. The model below is randomly initialized; load a checkpoint before this cell if you want trained predictions.

In [ ]:
from hippy2d.models import PrototypeOptimal

torch.manual_seed(42)

model_config = {
    "in_channels": 1,
    "input_size": 224,
    "kernels_size": [11, 11, 11, 11],
    "layers": [1, 2, 2, 2],
    "channels": [16, 20, 26, 32],
    "drop_rate": 0.2,
}

model = PrototypeOptimal(**model_config)
model.eval()

sum(p.numel() for p in model.parameters())

## Load And Mirror The Test Image

In [ ]:
from hippy2d.utils import get_testing_img

target_size = model_config["input_size"]

img = get_testing_img(rgb=False).convert("L").resize((target_size, target_size))
mirror_lr = ImageOps.mirror(img)
mirror_tb = ImageOps.flip(img)

def image_to_tensor(image):
    arr = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(arr)[None, None]

images = {
    "original": image_to_tensor(img),
    "left-right mirror": image_to_tensor(mirror_lr),
    "top-bottom mirror": image_to_tensor(mirror_tb),
}

batch = torch.cat(list(images.values()), dim=0)
batch.shape

## Pixel-Space Differences

In [ ]:
original = images["original"][0, 0]
lr = images["left-right mirror"][0, 0]
tb = images["top-bottom mirror"][0, 0]

diff_original_lr = (original - lr).abs()
diff_original_tb = (original - tb).abs()
diff_lr_tb = (lr - tb).abs()

fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)
panels = [
    (original, "Original", "gray"),
    (lr, "Left-right mirror", "gray"),
    (tb, "Top-bottom mirror", "gray"),
    (diff_original_lr, "|Original - LR|", "magma"),
    (diff_original_tb, "|Original - TB|", "magma"),
    (diff_lr_tb, "|LR - TB|", "magma"),
]

for ax, (panel, title, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(panel, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    if cmap != "gray":
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

## Model-Output Differences

In [ ]:
with torch.inference_mode():
    logits = model(batch)

logits_np = logits.detach().cpu().numpy()
names = list(images.keys())

pairwise = np.zeros((len(names), len(names)), dtype=np.float32)
for i in range(len(names)):
    for j in range(len(names)):
        pairwise[i, j] = np.linalg.norm(logits_np[i] - logits_np[j])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].plot(logits_np.T, marker="o")
axes[0].set_title("Logits by image version")
axes[0].set_xlabel("Class index")
axes[0].set_ylabel("Logit")
axes[0].legend(names)

im = axes[1].imshow(pairwise, cmap="viridis")
axes[1].set_title("Pairwise logit L2 distance")
axes[1].set_xticks(range(len(names)), names, rotation=30, ha="right")
axes[1].set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        axes[1].text(j, i, f"{pairwise[i, j]:.3g}", ha="center", va="center", color="white")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.show()

pairwise